In [ ]:
import os
import re
import glob
import copy
import enum
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

plt.close("all")

In [ ]:
input_folder = "../build/cutsPower__2023-08-07T23-59-34"

In [ ]:
tolerance: float = 0.0000000001
quantile: float = 0.99
max_value: float = 100000000000000000000000000000.0

class CutType(enum.Enum):
    AllForce = "all(force)"
    All = "all"
    MySplit = "my(split)"
    MyMIR = "my(mir)"
    MySplitAll = "my(split)+all"
    MyMIRAll = "my(mir)+all"
    NoCuts = "none"
    Clique = "clique"
    Gomory = "gomory"
    MIR = "mir"
    LiftPr = "liftp"
    ZeroH = "zeroh"
    Disj = "disj"
    GUB = "gub"
    Cover = "cover"
    FlowC = "flowc"
    FlowP = "flowp"
    MCF = "mcf"
    ImplL = "impliedl"
    ImplG = "impliedg"

CutColors = {
CutType.AllForce.value : "green",
CutType.All.value : "green",
CutType.MySplit.value : "red",
CutType.MyMIR.value : "red",
CutType.MySplitAll.value : "green",
CutType.MyMIRAll.value : "green",
CutType.NoCuts.value : "black",
CutType.Clique.value : "blue",
CutType.Gomory.value : "blue",
CutType.MIR.value : "blue",
CutType.LiftPr.value : "blue",
CutType.ZeroH.value : "blue",
CutType.Disj.value : "blue",
CutType.GUB.value : "blue",
CutType.Cover.value : "blue",
CutType.FlowC.value : "blue",
CutType.FlowP.value : "blue",
CutType.MCF.value : "blue",
CutType.ImplL.value : "blue",
CutType.ImplG.value : "blue"
}

class GroupType(enum.Enum):
    Cut = "cut_type"
    LPDiff = "lp_diff"
    MILPDiff = "milp_diff"
    CutoffDiff = "cutoff_diff"
    

In [ ]:
cut_type_pattern = r"Cut-type =\s*\"(?P<" + GroupType.Cut.value + r">[^\"]+(?=\":))"
lp_diff_pattern = r"LP Diff. =\s*(?P<" + GroupType.LPDiff.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
milp_diff_pattern = r"MILP Diff. =\s*(?P<" + GroupType.MILPDiff.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
cutoff_diff_pattern = r"Cutoff Diff. =\s*(?P<" + GroupType.CutoffDiff.value + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"

In [ ]:
def parse_file(filename : str) -> {str : float}:
    results : {str : float} = {}
    lines_counter : int = 0
    with open(filename, "r") as file:
        for line in file:
            # if not line.strip(): 
            #     continue
            cut_type: str = ""
            lp_diff: float = 0.0 

            match = re.search(cut_type_pattern, line)
            if match:
                cut_type = match.groupdict()[GroupType.Cut.value]
            elif line != "\n":
                raise Exception(f"Can't math the cut type in {lines_counter}-th line of {filename}")

            match = re.search(lp_diff_pattern, line)
            if match:
                lp_diff = float(match.groupdict()[GroupType.LPDiff.value])
            elif line != "\n":
                raise Exception(f"Can't math the LP-diff in {lines_counter}-th line of {filename}")

            if cut_type != "":
                results[cut_type] = lp_diff

            lines_counter += 1

    return results

In [ ]:
def make_patterns_list():
    # note: regex used for number with exponent: [-+]*\d*\.*\d+([eE][-+]?\d+)?
    pattern_cut_type = r"Cut-type =\s*\"(?P<" + GroupType.Cut + r">[^\"]+(?=\":))"
    pattern_lp_diff = r"LP Diff. =\s*(?P<" + GroupType.LPDiff + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
    pattern_milp_diff = r"MILP Diff. =\s*(?P<" + GroupType.MILPDiff + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"
    pattern_cutoff_diff = r"Cutoff Diff. =\s*(?P<" + GroupType.CutoffDiff + r">[-+]*\d*\.*\d+([eE][-+]?\d+)?)"

    return (
        (pattern_cut_type, GroupType.Cut),
        (pattern_lp_diff, GroupType.LPDiff),
        (pattern_milp_diff, GroupType.MILPDiff),
        (pattern_cutoff_diff, GroupType.CutoffDiff)
    )

In [ ]:
def parse_logs(logs_path: str) -> pd.DataFrame:
    
    problem_names : [str] = []
    df_rows : [{str : float}] = []

    for filename in glob.glob(f"{logs_path}/*gz_results.txt"):
        problem_name : str = re.search(r"(?<=/)[^/]+(?=.mps.gz_results.txt)", filename)[0]
        problem_names.append(problem_name)

        problem_data : {str : float} = parse_file(filename)

        df_rows.append(problem_data)

    return pd.DataFrame(df_rows, index=problem_names)

pure_data_df = parse_logs(input_folder)
# pure_data_df

In [ ]:
def filter_fun(x: float):
    x = x if x > tolerance else 0.0
    x = x if x < max_value else max_value

    return x

filtered_df = pure_data_df.applymap(filter_fun)
filtered_df = filtered_df[(filtered_df != 0.0).any(axis="columns")]

problems_number = len(filtered_df.index)

filtered_df

In [ ]:
relative_df = filtered_df.div(filtered_df["all(force)"].values, axis=0)
relative_df -= 1.0
relative_df.loc["MEAN"] = relative_df.mean(axis="index")
relative_df = relative_df.drop("all(force)", axis="columns")
relative_df

In [ ]:
# def make_quantile_filtering(df : pd.DataFrame) -> pd.DataFrame:
#     none_column = df[CutType.NoCuts.value]

# relative_df = make_relative_df(tolerance_filtered_df)

In [ ]:
# plt.figure()

# tolerance_filtered_df.plot(figsize=(35,35), kind="bar")

# filtered_df.plot.bar(figsize=(40,400))

In [ ]:
fig, axes = plt.subplots(nrows=problems_number, ncols=2, figsize=(1*len(CutType), 4 * problems_number))

plt.subplots_adjust(wspace=0.2, hspace=1)

for i, name in enumerate(filtered_df.index):
    axes[i,0].set_title(name)
    axes[i,1].set_title(name + " -- all(force) = 100%")
    
    filtered_df.loc[name].plot.bar(ax=axes[i,0], sharex = False, sharey = False, color=[CutColors[key] for key in filtered_df.columns])

    relative_problem_df = relative_df.loc[name]

    plt = relative_problem_df.plot.bar(ax=axes[i,1], sharex = False, sharey = False, color=[CutColors[key] for key in relative_problem_df.index])
    plt.axhline(y=0, linewidth=1, linestyle="--", color='r')

In [ ]:
mean_rel_df = relative_df.loc[name]

plt = mean_rel_df.plot.bar(color=[CutColors[key] for key in mean_rel_df.index])
plt.axhline(y=0, linewidth=1, linestyle="--", color='r')